# Pure Reconstruction Autoencoder — Centroid-Based Face Recognition

**20 Identities × 10 Images | Embedding-Only Recognition (No Classification Head)**

Pipeline:
1. Train autoencoder on ALL LFW faces (reconstruction only — no labels)
2. Select 20 identities, enroll by computing mean embeddings (cluster centers)
3. Recognize test faces by nearest centroid — **no retraining needed**
4. Demo: enroll a 21st person without retraining

In [ ]:
import os, warnings, random
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
from glob import glob

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print("All modules imported successfully.")

## Configuration

In [ ]:
# ── Image & Training ──
IMG_SIZE      = 128
BATCH_SIZE    = 32
AE_EPOCHS     = 50
LEARNING_RATE = 1e-3

# ── Recognition Setup ──
N_IDENTITIES  = 20        # number of people for recognition
IMGS_PER_ID   = 10        # images per person
N_ENROLL      = 7         # images used for enrollment (computing centroids)
N_TEST        = 3         # images used for testing

# ── Embedding ──
Z_DIM         = 128       # latent embedding dimension

# ── Dataset ──
DATA_DIR = "/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"

print(f"Recognition setup: {N_IDENTITIES} identities, {IMGS_PER_ID} images each")
print(f"Split: {N_ENROLL} enrollment + {N_TEST} test per person")
print(f"Embedding dimension: {Z_DIM}")

## Dataset Preparation
### Step 1: Select 20 Identities (≥10 images each)

In [ ]:
all_identities = {}
for person_dir in sorted(os.listdir(DATA_DIR)):
    person_path = os.path.join(DATA_DIR, person_dir)
    if os.path.isdir(person_path):
        imgs = sorted(glob(os.path.join(person_path, "*.jpg")))
        if len(imgs) >= IMGS_PER_ID:
            all_identities[person_dir] = imgs

print(f"Identities with >= {IMGS_PER_ID} images: {len(all_identities)}")

# Pick top-20 by number of available images
sorted_ids = sorted(all_identities.keys(),
                    key=lambda x: len(all_identities[x]), reverse=True)
selected_ids = sorted_ids[:N_IDENTITIES]

# Take exactly IMGS_PER_ID per person
recognition_data = {}
for person in selected_ids:
    recognition_data[person] = all_identities[person][:IMGS_PER_ID]

print(f"\nSelected {N_IDENTITIES} identities:")
for i, p in enumerate(selected_ids):
    print(f"  {i:2d}. {p}  ({len(all_identities[p])} avail, using {IMGS_PER_ID})")

### Step 2: Enrollment / Test Split

In [ ]:
label_map = {p: i for i, p in enumerate(selected_ids)}

enroll_paths = {}
test_paths   = {}

for person in selected_ids:
    imgs = list(recognition_data[person])   # copy
    np.random.shuffle(imgs)
    enroll_paths[person] = imgs[:N_ENROLL]
    test_paths[person]   = imgs[N_ENROLL:N_ENROLL + N_TEST]

total_enroll = sum(len(v) for v in enroll_paths.values())
total_test   = sum(len(v) for v in test_paths.values())
print(f"Enrollment: {total_enroll} images  ({N_ENROLL}/person)")
print(f"Test:       {total_test} images  ({N_TEST}/person)")

### Step 3: Load ALL LFW for Autoencoder Training

The autoencoder trains on **all** LFW faces for reconstruction — it never sees identity labels.

In [ ]:
all_img_paths = []
for person_dir in os.listdir(DATA_DIR):
    pdir = os.path.join(DATA_DIR, person_dir)
    if os.path.isdir(pdir):
        all_img_paths.extend(glob(os.path.join(pdir, "*.jpg")))

np.random.shuffle(all_img_paths)
n_val = int(0.1 * len(all_img_paths))
train_paths_ae = all_img_paths[n_val:]
val_paths_ae   = all_img_paths[:n_val]

print(f"Total LFW images:   {len(all_img_paths)}")
print(f"AE train images:    {len(train_paths_ae)}")
print(f"AE val images:      {len(val_paths_ae)}")

In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, img                       # input == target for AE

def make_ds(paths, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices(paths)
    if shuffle:
        ds = ds.shuffle(len(paths))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train_paths_ae)
val_ds   = make_ds(val_paths_ae, shuffle=False)
print("tf.data pipelines ready.")

## Autoencoder Architecture

4-layer convolutional encoder → 128-dim bottleneck → 4-layer decoder.  
Trained purely on reconstruction (MSE). No classification head.

In [ ]:
def build_autoencoder():
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='encoder_input')
    x = inp

    for i, f in enumerate([32, 64, 64, 64]):
        x = layers.Conv2D(f, 3, strides=2, padding='same',
                          name=f'enc_conv_{i}')(x)
        x = layers.BatchNormalization(name=f'enc_bn_{i}')(x)
        x = layers.Activation('relu')(x)

    shape_before = x.shape[1:]            # (8, 8, 64)
    x = layers.Flatten()(x)
    embedding = layers.Dense(Z_DIM, name='embedding')(x)

    # ── decoder ──
    x = layers.Dense(int(np.prod(shape_before)), name='dec_dense')(embedding)
    x = layers.Reshape(shape_before)(x)

    for i, f in enumerate([64, 64, 32, 3]):
        x = layers.Conv2DTranspose(f, 3, strides=2, padding='same',
                                   name=f'dec_convT_{i}')(x)
        if i < 3:
            x = layers.BatchNormalization(name=f'dec_bn_{i}')(x)
            x = layers.Activation('relu')(x)
        else:
            x = layers.Activation('sigmoid')(x)

    ae  = Model(inp, x, name='Face_Autoencoder')
    enc = Model(inp, embedding, name='Encoder')
    return ae, enc

autoencoder, encoder = build_autoencoder()
autoencoder.summary()
print(f"\nEncoder output shape: {encoder.output_shape}")
print(f"Total params: {autoencoder.count_params():,}")

## Phase 1 — Train Autoencoder (Reconstruction Only)

No identity labels used. The encoder learns a general face representation from reconstruction alone.

In [ ]:
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse'
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

print(f"Training for up to {AE_EPOCHS} epochs on {len(train_paths_ae)} images …")
print("=" * 60)

history = autoencoder.fit(
    train_ds,
    validation_data=val_ds,
    epochs=AE_EPOCHS,
    callbacks=callbacks
)

### Reconstruction Loss Curves

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(history.history['loss'], 'r-',  label='Train', linewidth=2)
plt.plot(history.history['val_loss'], 'b--', label='Val', linewidth=2)
plt.title('Reconstruction Loss (MSE)', fontsize=14)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(fontsize=12); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Final  train loss: {history.history['loss'][-1]:.6f}")
print(f"Final  val   loss: {history.history['val_loss'][-1]:.6f}")

### Reconstruction Samples

In [ ]:
sample_imgs = next(iter(val_ds))[0][:10]
recon = autoencoder.predict(sample_imgs, verbose=0)

plt.figure(figsize=(20, 4))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(sample_imgs[i].numpy()); plt.axis('off')
    plt.title('Original', fontsize=8)
    plt.subplot(2, 10, i + 11)
    plt.imshow(np.clip(recon[i], 0, 1)); plt.axis('off')
    plt.title('Reconstructed', fontsize=8)
plt.suptitle('Autoencoder Reconstruction Quality', fontsize=13)
plt.tight_layout(); plt.show()

## Centroid-Based Face Recognition

### Step 1 — Enroll: Compute Cluster Centers

In [ ]:
def get_embedding(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return encoder.predict(tf.expand_dims(img, 0), verbose=0)[0]

print("Computing enrollment embeddings …")
enroll_embs = {}                         # person → (N_ENROLL, Z_DIM)
for person in selected_ids:
    embs = [get_embedding(p) for p in enroll_paths[person]]
    enroll_embs[person] = np.stack(embs)

# Cluster centers = mean embedding per person
centers = {p: np.mean(enroll_embs[p], axis=0) for p in selected_ids}
print(f"Computed {len(centers)} cluster centers  (each {Z_DIM}-D)")

### Step 2 — Visualize Embedding Clusters (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

all_e, all_l = [], []
for p in selected_ids:
    for e in enroll_embs[p]:
        all_e.append(e); all_l.append(label_map[p])
all_e = np.array(all_e); all_l = np.array(all_l)

center_arr = np.array([centers[p] for p in selected_ids])
combined = np.vstack([all_e, center_arr])
proj = TSNE(n_components=2, random_state=42,
            perplexity=min(30, len(combined) - 1)).fit_transform(combined)

emb_proj = proj[:len(all_e)]
ctr_proj = proj[len(all_e):]

plt.figure(figsize=(12, 10))
cmap = plt.cm.tab20(np.linspace(0, 1, N_IDENTITIES))
for i, p in enumerate(selected_ids):
    mask = all_l == i
    plt.scatter(emb_proj[mask, 0], emb_proj[mask, 1],
                c=[cmap[i]], label=p[:15], alpha=0.7, s=60)
    plt.scatter(ctr_proj[i, 0], ctr_proj[i, 1],
                c=[cmap[i]], marker='X', s=250,
                edgecolors='black', linewidth=1.5)

plt.title(f't-SNE of Encoder Embeddings ({N_IDENTITIES} Identities)', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

### Step 3 — Recognize Test Images by Nearest Centroid

In [ ]:
from scipy.spatial.distance import euclidean

print("Running centroid-based recognition on test set …")
print("=" * 60)

correct1 = 0; correct5 = 0; total = 0
results = []

for true_person in selected_ids:
    for path in test_paths[true_person]:
        emb = get_embedding(path)
        dists = {p: euclidean(emb, c) for p, c in centers.items()}
        ranked = sorted(dists.items(), key=lambda x: x[1])

        pred = ranked[0][0]
        top5 = [r[0] for r in ranked[:5]]

        hit1 = pred == true_person
        hit5 = true_person in top5
        correct1 += int(hit1); correct5 += int(hit5); total += 1

        results.append(dict(true=true_person, pred=pred,
                            correct=hit1, path=path, dists=dists))

top1_acc = correct1 / total * 100
top5_acc = correct5 / total * 100

print(f"\nCentroid Recognition  ({N_IDENTITIES} identities, {total} test images)")
print(f"  Top-1 Accuracy:  {correct1}/{total}  =  {top1_acc:.2f}%")
print(f"  Top-5 Accuracy:  {correct5}/{total}  =  {top5_acc:.2f}%")

### Per-Identity Breakdown

In [ ]:
print(f"{'Identity':<30} {'Correct':>7} {'Total':>6} {'Acc':>8}")
print("-" * 55)
for p in selected_ids:
    pr = [r for r in results if r['true'] == p]
    nc = sum(r['correct'] for r in pr)
    print(f"{p:<30} {nc:>7} {len(pr):>6} {nc/len(pr)*100:>7.1f}%")

### Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_true = [label_map[r['true']] for r in results]
y_pred = [label_map[r['pred']] for r in results]
cm = confusion_matrix(y_true, y_pred, labels=list(range(N_IDENTITIES)))

plt.figure(figsize=(14, 12))
short = [p[:12] for p in selected_ids]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short, yticklabels=short)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion Matrix — Centroid Recognition ({N_IDENTITIES} IDs)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

### Visual Recognition Demo

In [ ]:
# Show 6 random test recognition results
show_idx = np.random.choice(len(results), min(6, len(results)), replace=False)

for idx in show_idx:
    r = results[idx]
    ranked = sorted(r['dists'].items(), key=lambda x: x[1])[:3]

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

    # Query image
    q_img = tf.image.decode_jpeg(tf.io.read_file(r['path']), channels=3)
    q_img = tf.image.resize(q_img, [IMG_SIZE, IMG_SIZE]) / 255.0
    axes[0].imshow(q_img.numpy()); axes[0].axis('off')
    mark = "CORRECT" if r['correct'] else "WRONG"
    axes[0].set_title(f"Query: {r['true'][:15]}\n({mark})", fontsize=10,
                      color='green' if r['correct'] else 'red')

    # Top-3 matches
    for j, (match_p, dist) in enumerate(ranked):
        rep = tf.image.decode_jpeg(tf.io.read_file(enroll_paths[match_p][0]), channels=3)
        rep = tf.image.resize(rep, [IMG_SIZE, IMG_SIZE]) / 255.0
        axes[j+1].imshow(rep.numpy()); axes[j+1].axis('off')
        c = 'green' if match_p == r['true'] else 'red'
        axes[j+1].set_title(f"#{j+1}: {match_p[:15]}\ndist={dist:.2f}",
                            fontsize=9, color=c)

    plt.tight_layout(); plt.show()

## Additional Classifiers on Embeddings

For comparison: KNN, SVM, Logistic Regression on the same encoder embeddings.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize

# Build feature matrices
X_enroll, y_enroll = [], []
for p in selected_ids:
    for e in enroll_embs[p]:
        X_enroll.append(e); y_enroll.append(label_map[p])
X_enroll = np.array(X_enroll); y_enroll = np.array(y_enroll)

X_test = np.array([get_embedding(r['path']) for r in results])
y_test = np.array([label_map[r['true']] for r in results])

X_enroll_n = normalize(X_enroll)
X_test_n   = normalize(X_test)

print(f"Enrollment: {X_enroll.shape}   Test: {X_test.shape}")
print("=" * 55)

# KNN
for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_enroll, y_enroll)
    print(f"KNN (k={k}):             {knn.score(X_test, y_test)*100:.2f}%")

# SVM Linear
svm = SVC(kernel='linear')
svm.fit(X_enroll_n, y_enroll)
print(f"SVM (linear):           {svm.score(X_test_n, y_test)*100:.2f}%")

# SVM RBF
svm2 = SVC(kernel='rbf')
svm2.fit(X_enroll_n, y_enroll)
print(f"SVM (RBF):              {svm2.score(X_test_n, y_test)*100:.2f}%")

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_enroll_n, y_enroll)
print(f"Logistic Regression:    {lr.score(X_test_n, y_test)*100:.2f}%")

# Centroid (already computed above)
print(f"Centroid Matching:      {top1_acc:.2f}%  (top-1)")
print(f"Centroid Matching:      {top5_acc:.2f}%  (top-5)")

## "21st Person" Enrollment Demo

Key advantage of the autoencoder: a new person can be enrolled by computing their mean embedding — **no retraining required**.

In [ ]:
# Find a person NOT in our 20 selected identities
other = [p for p in all_identities.keys() if p not in selected_ids]
new_person = other[0]
new_imgs = all_identities[new_person][:IMGS_PER_ID]

# Use first 7 for enrollment, last 3 for test (same split ratio)
new_enroll = new_imgs[:N_ENROLL]
new_test   = new_imgs[N_ENROLL:N_ENROLL + N_TEST]

print(f"21st person: {new_person}")
print(f"  Enrollment images: {len(new_enroll)}")
print(f"  Test images:       {len(new_test)}")

# Compute center for the new person
new_center = np.mean([get_embedding(p) for p in new_enroll], axis=0)

# Expanded center database (21 people)
centers_21 = {**centers, new_person: new_center}

print(f"\nCluster centers: {len(centers)} → {len(centers_21)}  (no retraining!)\n")

# Test the new person
print(f"--- Testing new person ({new_person}) ---")
for path in new_test:
    emb = get_embedding(path)
    dists = {p: euclidean(emb, c) for p, c in centers_21.items()}
    ranked = sorted(dists.items(), key=lambda x: x[1])
    pred = ranked[0][0]
    mark = "CORRECT" if pred == new_person else "WRONG"
    print(f"  Predicted: {pred:<25}  ({mark})")

# Test an existing person still works with 21 centers
print(f"\n--- Testing existing person ({selected_ids[0]}) with 21 centers ---")
for path in test_paths[selected_ids[0]]:
    emb = get_embedding(path)
    dists = {p: euclidean(emb, c) for p, c in centers_21.items()}
    ranked = sorted(dists.items(), key=lambda x: x[1])
    pred = ranked[0][0]
    mark = "CORRECT" if pred == selected_ids[0] else "WRONG"
    print(f"  Predicted: {pred:<25}  ({mark})")

In [ ]:
# Visual: show new person enrollment + recognition
fig, axes = plt.subplots(2, max(N_ENROLL, 4), figsize=(20, 6))

# Row 1: enrollment images
for i in range(N_ENROLL):
    img = tf.image.decode_jpeg(tf.io.read_file(new_enroll[i]), channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0
    axes[0, i].imshow(img.numpy()); axes[0, i].axis('off')
    axes[0, i].set_title(f'Enroll {i+1}', fontsize=9)
for i in range(N_ENROLL, axes.shape[1]):
    axes[0, i].axis('off')

# Row 2: test images with predictions
for i in range(min(N_TEST, axes.shape[1])):
    if i < len(new_test):
        img = tf.image.decode_jpeg(tf.io.read_file(new_test[i]), channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0
        emb = get_embedding(new_test[i])
        dists = {p: euclidean(emb, c) for p, c in centers_21.items()}
        pred = min(dists, key=dists.get)
        axes[1, i].imshow(img.numpy()); axes[1, i].axis('off')
        c = 'green' if pred == new_person else 'red'
        axes[1, i].set_title(f'Test → {pred[:15]}', fontsize=9, color=c)
for i in range(N_TEST, axes.shape[1]):
    axes[1, i].axis('off')

plt.suptitle(f'21st Person Demo: {new_person}\n(Top: enrollment, Bottom: test predictions)',
             fontsize=13)
plt.tight_layout(); plt.show()

## Summary

In [ ]:
print("=" * 60)
print("PURE RECONSTRUCTION AE — CENTROID RECOGNITION RESULTS")
print("=" * 60)
print(f"\nArchitecture:  4-layer Conv AE, {Z_DIM}-dim embedding, BN")
print(f"Training:      MSE reconstruction on {len(train_paths_ae)} LFW images")
print(f"Recognition:   {N_IDENTITIES} identities, {N_ENROLL} enroll + {N_TEST} test each")
print(f"Total params:  {autoencoder.count_params():,}")
print(f"\n{'Metric':<25} {'Value':>10}")
print("-" * 37)
print(f"{'Centroid Top-1':<25} {top1_acc:>9.2f}%")
print(f"{'Centroid Top-5':<25} {top5_acc:>9.2f}%")
print(f"\nKey insight: No classification head, no identity labels during")
print(f"training. Recognition is purely distance-based on encoder")
print(f"embeddings. New identities enrolled without retraining.")